# RSNA Knee Abnormality Detection: frozen DINOv2 features with a weak-label ensemble

Multi-label classification of 12 knee MRI findings (macro-averaged ROC-AUC) in a Kaggle code competition. Only 58 of 4,407 training studies carry direct labels; the rest come with free-text, multilingual radiology reports. The scored run executes this notebook offline against the hidden test set.

**Approach**

1. Select one series per anatomical plane, read three physically ordered slices per plane, crop to a fixed field of view in millimetres, and mirror axial/coronal images to a canonical knee side.
2. Extract frozen DINOv2 (ViT-S/14) features, mean-pooled over slices and concatenated across planes, and cache them.
3. Derive weak labels from the reports with a TF-IDF classifier trained on the 58 gold studies; gold labels override them.
4. Split by report text with multi-label stratification so duplicate reports never straddle folds and rare findings appear in every fold.
5. Train a small MLP head in two phases: a weak-plus-gold warm-up, then a gold-only fine-tune.
6. Build a three-group ensemble (frozen heads, last-block fine-tuned backbones, LoRA backbones), average within groups, combine with fixed weights, and write rank-normalised predictions.

Design choices are justified inline with results from leave-one-out experiments on the gold studies; the tables are collected in `docs/experiments.md`.

## 1. Configuration

All tunable settings live in one block. Cache filenames embed the settings that determine their contents (image size, crop, slice count, laterality flag), so changing a setting never reuses a stale cache.

In [1]:
CONFIG = {
    # --- imaging ---
    "IMG_SIZE": 224,            # square input size fed to the backbone
    "CROP_MM": 120.0,           # physical field of view cropped from the centre of each slice
    "SLICES_PER_PLANE": 3,      # slices sampled per selected series, spread through the stack
    "SLICE_QUANTILES": (0.35, 0.50, 0.65),  # where in the physically-ordered stack to sample
    "LATERALITY_CORRECTION": True,  # mirror axial/coronal images so left and right knees align

    # --- backbone ---
    "BACKBONE_NAME": "vit_small_patch14_dinov2",
    "PRETRAINED": True,         # set False only for a quick mechanics smoke test
    "LOCAL_BACKBONE_CHECKPOINT": "/kaggle/input/dinov2-weights/dinov2_backbone.pth",  # weights saved from a first run and attached as a dataset; None downloads from the Hub
    "FREEZE_BACKBONE": True,    # frozen feature extraction; partial fine-tuning is handled per ensemble member

    # --- partial backbone fine-tuning (ensemble members) ---
    "N_TRAINABLE_BACKBONE_BLOCKS": 1,   # last transformer blocks to unfreeze; 1 overfit less than 2 (docs/experiments.md)
    "BACKBONE_FINETUNE_LR": 1e-5,       # small learning rate for the unfrozen backbone parameters
    "BACKBONE_FINETUNE_EPOCHS": 80,
    "BACKBONE_FINETUNE_BATCH_SIZE": 8,  # small: each study is 3 planes x SLICES_PER_PLANE images
    "INCLUDE_BACKBONE_FINETUNE_IN_SUBMISSION": True,  # add fully fine-tuned members to the ensemble
    "FINETUNE_SEEDS": [1, 2],  # random seeds averaged per fold, to dampen seed-dependent noise

    # --- light augmentation during backbone fine-tuning ---
    "AUGMENT_FINETUNING": True,      # applied to every fine-tuned and LoRA member (no measurable change in leave-one-out CV)
    "AUGMENT_ROTATION_DEG": 5.0,     # small-angle rotation: plausible patient-positioning noise
    "AUGMENT_TRANSLATE_FRAC": 0.04,  # translation, as a fraction of IMG_SIZE
    "AUGMENT_INTENSITY_SCALE": 0.10, # +/- intensity scale/bias jitter, mimics scanner-to-scanner variation
    "AUGMENT_NOISE_STD": 0.02,       # mild Gaussian noise added after the intensity jitter
    "AUGMENT_FLIP_PROB": 0.0,        # off: a random flip would undo the deterministic laterality mirror
    "AUGMENT_SEED": 777,             # separate RNG stream, offset by each member's seed

    # --- LoRA members ---
    "N_LORA_BLOCKS": 4,    # adapters on the attention projections of the last N blocks
    "LORA_RANK": 8,
    "LORA_ALPHA": 16,      # the low-rank update is scaled by LORA_ALPHA / LORA_RANK
    "LORA_LR": 1e-4,

    # --- ensemble ---
    "INCLUDE_LORA_MEMBERS": True,
    "LORA_SEEDS": [1, 2],         # LoRA members per fold = len(LORA_SEEDS)
    "EXTRA_FROZEN_HEAD_SEEDS": [11, 12],  # additional frozen-head seeds per fold (cheap)
    "ENSEMBLE_GROUP_WEIGHTS": {"frozen": 0.5, "finetuned": 0.25, "lora": 0.25},  # a hedge toward the best single group (frozen), not a tuned optimum
    "ENSEMBLE_TIME_BUDGET_SEC": 6 * 3600,   # optional members are skipped once this much wall-clock has elapsed
    "TEST_PREDICT_CHUNK": 32,     # studies per forward pass when predicting the test set through a member
    "SAVE_ENSEMBLE_WEIGHTS": False,  # True also writes member weights to /kaggle/working/ensemble/
    "CLEAN_CACHE_AT_END": True,   # delete CACHE_DIR at the end so the saved version stays small

    # --- data pipeline ---
    "N_IO_WORKERS": 16,         # thread pool size for parallel DICOM reading
    "FEATURE_BATCH_SIZE": 64,   # batch size used only for the backbone forward pass

    # --- head training ---
    "HEAD_HIDDEN": 256,
    "HEAD_DROPOUT": 0.3,
    "PHASE1_EPOCHS": 40,         # short warm-up on the full weak+gold weighted pool
    "PHASE1_LR": 1e-3,
    "PHASE2_EPOCHS": 200,        # fine-tune using only the directly labelled studies
    "PHASE2_LR": 1e-4,
    "HEAD_WEIGHT_DECAY": 1e-4,
    "HEAD_BATCH_SIZE": 128,

    # --- validation ---
    "N_FOLDS": 5,
    "VALID_FOLD": 0,

    # --- caching ---
    "CACHE_DIR": "/kaggle/working/cache",
}

import os, time
NOTEBOOK_START = time.time()
os.makedirs(CONFIG["CACHE_DIR"], exist_ok=True)
print("configuration set. slices per study per plane:", CONFIG["SLICES_PER_PLANE"])

configuration set. slices per study per plane: 3


## 2. Data and series selection

The competition root is found by walking `/kaggle/input`. For each plane, a fluid-sensitive series is preferred when one exists, since those sequences are informative for more of the twelve findings than a purely structural one.

In [2]:
import re
import time
import hashlib
import unicodedata
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
torch.manual_seed(0)
np.random.seed(0)

def find_competition_root(base="/kaggle/input"):
    for root, dirs, files in os.walk(base):
        if "train.csv" in files and ("train_series.csv" in files or "train_series" in dirs):
            return root
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    raise FileNotFoundError(
        "Could not locate the competition files under /kaggle/input. "
        "Attach the RSNA Knee Abnormality Detection dataset to this notebook."
    )

ROOT = Path(find_competition_root())
train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")

TARGETS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
has_full_labels = train[TARGETS].notna().all(axis=1)

print(f"studies total     : {len(train):,}  (directly labelled: {int(has_full_labels.sum())})")
print(f"targets           : {TARGETS}")
print(f"device            : {DEVICE}")

studies total     : 4,407  (directly labelled: 58)
targets           : ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
device            : cuda


In [3]:
def select_series_per_plane(series_df):
    # Returns, for every study, the chosen SeriesInstanceUID for each of the three planes.
    selected = {}
    fluid_col = "Fluid_Sensitive" if "Fluid_Sensitive" in series_df.columns else None
    for study_id, group in series_df.groupby("StudyInstanceUID"):
        choice = {}
        for plane in ("Sagittal", "Coronal", "Axial"):
            candidates = group[group["Anatomical_Plane"] == plane]
            if candidates.empty:
                continue
            ranked = candidates.sort_values(fluid_col, ascending=False) if fluid_col else candidates
            choice[plane] = ranked.iloc[0]["SeriesInstanceUID"]
        selected[study_id] = choice
    return selected

train_plane_series = select_series_per_plane(train_series)
test_plane_series = select_series_per_plane(test_series)

n_with_all_three = sum(1 for c in train_plane_series.values() if len(c) == 3)
print(f"training studies with all three planes available : {n_with_all_three}/{len(train_plane_series)}")

training studies with all three planes available : 4407/4407


## 3. Slice extraction

- **Ordering:** files in a series are sorted by physical position (`ImagePositionPatient`, `ImageOrientationPatient`), not by filename.
- **Scale:** each slice is cropped to a fixed physical field of view using the DICOM pixel spacing, then resized, so anatomy has a comparable scale across scanners.
- **Sampling:** `SLICES_PER_PLANE` slices are taken at fixed quantiles of each ordered stack. DICOM reading is I/O bound, so a thread pool is used.
- **Laterality:** five findings (medial/lateral meniscus, medial/lateral OA, MCL) are defined relative to the body midline, so "medial" lands on opposite sides of the image for left and right knees. Laterality comes from the DICOM `Laterality` tag, falling back to the sign of the slice x-coordinate (LPS convention), and axial and coronal images are mirrored to one canonical side.

  *Evidence:* in a leave-one-out probe on the gold studies, correction moved the side-dependent findings from 0.614 to 0.619 macro-AUC and the side-independent controls from 0.601 to 0.602. The effect is within noise, so the correction is kept because it is physically motivated and harmless, not because it measurably helps.

  *Sagittal plane left uncorrected:* there, medial versus lateral corresponds to which slice is viewed, so fixing it means reversing the sampling order, and the required sign depends on vendor-specific orientation conventions that could not be verified. A silently wrong correction would be worse than none.

Slices are cached on disk as uint8 (a loss of at most 1/255 on already-normalised data), cutting the cache from about 7.6 GB to 1.9 GB. Quantisation is done per study inside the read loop; a single array-wide conversion caused a memory spike large enough to restart the kernel.

In [4]:
def determine_study_laterality(study_id, choice, series_root):
    # Returns "L", "R", or None, using the DICOM Laterality tag where present and falling
    # back to the sign of a representative slice's physical x-position (LPS convention:
    # positive x is the patient's left) where it is not.
    for plane in ("Axial", "Coronal", "Sagittal"):
        series_id = choice.get(plane)
        if series_id is None:
            continue
        series_dir = os.path.join(series_root, str(study_id), str(series_id))
        try:
            files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
            if not files:
                continue
            ds = pydicom.dcmread(os.path.join(series_dir, files[0]), stop_before_pixels=True,
                                  force=True, specific_tags=["Laterality", "ImageLaterality"])
            for tag in ("Laterality", "ImageLaterality"):
                value = str(getattr(ds, tag, "")).strip().upper()[:1]
                if value in ("L", "R"):
                    return value
        except Exception:
            continue

    for plane in ("Axial", "Coronal", "Sagittal"):
        series_id = choice.get(plane)
        if series_id is None:
            continue
        series_dir = os.path.join(series_root, str(study_id), str(series_id))
        try:
            files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
            if not files:
                continue
            mid_file = files[len(files) // 2]
            ds = pydicom.dcmread(os.path.join(series_dir, mid_file), stop_before_pixels=True,
                                  force=True, specific_tags=["ImagePositionPatient"])
            x = float(ds.ImagePositionPatient[0])
            if abs(x) < 5.0:   # too close to the midline for the sign to be a confident signal
                continue
            return "L" if x > 0 else "R"
        except Exception:
            continue
    return None

def physically_ordered_files(series_dir):
    files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
    keyed = []
    for f in files:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f), stop_before_pixels=True, force=True,
                                  specific_tags=["ImagePositionPatient", "ImageOrientationPatient"])
            orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
            position = np.asarray(ds.ImagePositionPatient, dtype=float)
            key = float(np.dot(position, np.cross(orientation[:3], orientation[3:])))
        except Exception:
            key = 0.0
        keyed.append((key, f))
    keyed.sort()
    return [f for _, f in keyed]

def read_slice(series_dir, filename, crop_mm, size, mirror=False):
    ds = pydicom.dcmread(os.path.join(series_dir, filename), force=True)
    array = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    array = array * slope + intercept

    try:
        spacing_mm_per_px = float(ds.PixelSpacing[0])
    except Exception:
        spacing_mm_per_px = crop_mm / max(array.shape)
    half_crop_px = max(1, int(round(crop_mm / spacing_mm_per_px / 2)))
    cy, cx = array.shape[0] // 2, array.shape[1] // 2
    y0, y1 = max(0, cy - half_crop_px), min(array.shape[0], cy + half_crop_px)
    x0, x1 = max(0, cx - half_crop_px), min(array.shape[1], cx + half_crop_px)
    crop = array[y0:y1, x0:x1]
    if crop.size == 0:
        crop = array

    low, high = np.percentile(crop, [1, 99])
    normalized = np.clip((crop - low) / max(high - low, 1e-6), 0, 1).astype(np.float32)
    y_idx = np.linspace(0, normalized.shape[0] - 1, size).astype(int)
    x_idx = np.linspace(0, normalized.shape[1] - 1, size).astype(int)
    resized = normalized[np.ix_(y_idx, x_idx)]
    return resized[:, ::-1] if mirror else resized

def study_slices(study_id, plane_series, series_root, cfg):
    # Returns an array of shape (3, K, size, size): plane x sampled-slice x H x W.
    # Missing planes, or a slice that fails to read, are returned as zero arrays rather than
    # raising, so a single bad file does not take down a whole batch.
    size = cfg["IMG_SIZE"]
    k = cfg["SLICES_PER_PLANE"]
    out = np.zeros((3, k, size, size), dtype=np.float32)
    choice = plane_series.get(study_id, {})

    laterality = None
    if cfg["LATERALITY_CORRECTION"]:
        laterality = determine_study_laterality(study_id, choice, series_root)

    for plane_idx, plane in enumerate(("Sagittal", "Coronal", "Axial")):
        series_id = choice.get(plane)
        if series_id is None:
            continue
        series_dir = os.path.join(series_root, str(study_id), str(series_id))
        # Only axial and coronal images are mirrored; see the markdown above for why the
        # sagittal plane is left uncorrected here. "L" is chosen arbitrarily as the side that
        # gets flipped, so every study ends up aligned to what a right knee looks like.
        mirror = cfg["LATERALITY_CORRECTION"] and laterality == "L" and plane in ("Axial", "Coronal")
        try:
            files = physically_ordered_files(series_dir)
            if not files:
                continue
            for slice_idx, q in enumerate(cfg["SLICE_QUANTILES"][:k]):
                file_idx = min(len(files) - 1, max(0, int(round(q * (len(files) - 1)))))
                out[plane_idx, slice_idx] = read_slice(series_dir, files[file_idx], cfg["CROP_MM"], size, mirror=mirror)
        except Exception:
            continue
    return out

def build_slice_cache(study_ids, plane_series, series_root, cfg, cache_name):
    # Reads (or loads from cache) the (N, 3, K, size, size) slice tensor for a list of studies.
    # The cache filename bakes in the settings that affect its contents, so switching
    # IMG_SIZE, CROP_MM, SLICES_PER_PLANE, or LATERALITY_CORRECTION after a first run does not
    # silently return a stale array computed under different settings. It also bakes in "_u8",
    # so a float32 cache from an older format is never mistaken for the uint8 one.
    #
    # On disk, slices are stored as uint8 (0-255) rather than float32: study_slices/read_slice
    # already normalize every slice to [0, 1], so the only information a uint8 round-trip loses
    # is at most 1/255 (~0.004), negligible next to that normalization's own percentile clipping.
    # This cuts this cache from about 1.7MB to about 0.43MB per study -- for the full training
    # set that is the difference between roughly 7.6GB and roughly 1.9GB of Kaggle's limited
    # /kaggle/working disk space, which is also spent on the backbone checkpoint and any saved
    # ensemble members. In memory, callers still see a float32 [0, 1] array.
    cache_key = (f"{cache_name}_sz{cfg['IMG_SIZE']}_crop{int(cfg['CROP_MM'])}"
                 f"_k{cfg['SLICES_PER_PLANE']}_lat{int(cfg['LATERALITY_CORRECTION'])}_u8")
    cache_path = os.path.join(cfg["CACHE_DIR"], f"{cache_key}.npy")
    if os.path.exists(cache_path):
        print(f"loading cached slices from {cache_path}")
        return np.load(cache_path).astype(np.float32) / 255.0

    size = cfg["IMG_SIZE"]
    k = cfg["SLICES_PER_PLANE"]
    result = np.zeros((len(study_ids), 3, k, size, size), dtype=np.float32)
    # Filled alongside result, one small per-study array at a time, rather than quantized from
    # result in one bulk operation afterward: result * 255.0, np.round(...), and np.clip(...)
    # each allocate a brand-new full-size float32 array if done on the whole array at once --
    # three more arrays the size of result (about 7.6GB each for the full training set) stacked
    # on top of result itself, which is still alive because it is returned below. That is a
    # transient 20-30GB+ spike for no durable reason, and is almost certainly what is forcing a
    # kernel restart. Quantizing per study instead means each temporary is a single study's
    # array (a few MB), never the whole dataset's.
    quantized = np.zeros((len(study_ids), 3, k, size, size), dtype=np.uint8)
    start = time.time()

    def _work(i):
        # A single study that fails for any unforeseen reason (a truly corrupted file, an
        # unusual DICOM structure neither anticipated nor covered by study_slices' own
        # internal guards, and so on) falls back to a blank placeholder here rather than
        # letting ThreadPoolExecutor re-raise it and take the whole submission down. Kaggle's
        # own code-competition guidance is explicit that the hidden test set can differ from
        # the public one, so a defensive fallback at this outermost level is worth having even
        # though study_slices already guards its own DICOM reads internally.
        try:
            return i, study_slices(study_ids[i], plane_series, series_root, cfg)
        except Exception as e:
            print(f"warning: study {study_ids[i]} failed unexpectedly ({type(e).__name__}: {e}); "
                  f"using a blank placeholder for it")
            size = cfg["IMG_SIZE"]
            k = cfg["SLICES_PER_PLANE"]
            return i, np.zeros((3, k, size, size), dtype=np.float32)

    with ThreadPoolExecutor(max_workers=cfg["N_IO_WORKERS"]) as pool:
        for count, (i, arr) in enumerate(pool.map(_work, range(len(study_ids)))):
            result[i] = arr
            quantized[i] = np.clip(np.round(arr * 255.0), 0, 255).astype(np.uint8)
            if (count + 1) % 500 == 0:
                print(f"  {count + 1}/{len(study_ids)} studies read ({time.time() - start:.0f}s elapsed)")

    np.save(cache_path, quantized)
    print(f"read {len(study_ids)} studies in {time.time() - start:.0f}s, cached to {cache_path} "
          f"({quantized.nbytes / 1e9:.2f}GB on disk)")
    return result

TRAIN_SERIES_ROOT = ROOT / "train_series"
TEST_SERIES_ROOT = ROOT / "test_series"

train_study_ids = train["StudyInstanceUID"].tolist()
test_study_ids = test["StudyInstanceUID"].tolist()

train_slices = build_slice_cache(train_study_ids, train_plane_series, TRAIN_SERIES_ROOT, CONFIG, "train_slices")
test_slices = build_slice_cache(test_study_ids, test_plane_series, TEST_SERIES_ROOT, CONFIG, "test_slices")
print("train_slices shape:", train_slices.shape, " test_slices shape:", test_slices.shape)

  500/4407 studies read (192s elapsed)
  1000/4407 studies read (382s elapsed)
  1500/4407 studies read (571s elapsed)
  2000/4407 studies read (753s elapsed)
  2500/4407 studies read (939s elapsed)
  3000/4407 studies read (1135s elapsed)
  3500/4407 studies read (1318s elapsed)
  4000/4407 studies read (1502s elapsed)
read 4407 studies in 1664s, cached to /kaggle/working/cache/train_slices_sz224_crop120_k3_lat1_u8.npy (1.99GB on disk)
read 3 studies in 5s, cached to /kaggle/working/cache/test_slices_sz224_crop120_k3_lat1_u8.npy (0.00GB on disk)
train_slices shape: (4407, 3, 3, 224, 224)  test_slices shape: (3, 3, 3, 224, 224)


Cache filenames include the settings that determine their content, so changing one produces a new file rather than silently reusing an old one.

## 4. Backbone and feature extraction

DINOv2 is loaded through `timm` with `num_classes=0` (pooled embedding) and `dynamic_img_size=True`, so inputs need not be 518 px; 224 px is used for speed. A frozen self-supervised backbone is used because 58 gold labels cannot support training a vision model from scratch, and it makes head iteration fast: features are extracted once and cached.

Grayscale slices are replicated to three channels. Slice embeddings are averaged within a plane and the three plane embeddings are concatenated (not averaged), since planes carry complementary information.

In [5]:
import timm

def load_backbone(cfg):
    model_kwargs = dict(num_classes=0, img_size=cfg["IMG_SIZE"], dynamic_img_size=True)
    if cfg["LOCAL_BACKBONE_CHECKPOINT"]:
        model = timm.create_model(cfg["BACKBONE_NAME"], pretrained=False, **model_kwargs)
        state_dict = torch.load(cfg["LOCAL_BACKBONE_CHECKPOINT"], map_location="cpu")
        model.load_state_dict(state_dict, strict=False)
        print(f"loaded backbone weights from local checkpoint: {cfg['LOCAL_BACKBONE_CHECKPOINT']}")
    else:
        model = timm.create_model(cfg["BACKBONE_NAME"], pretrained=cfg["PRETRAINED"], **model_kwargs)
        if cfg["PRETRAINED"]:
            print("loaded pretrained backbone weights from the Hugging Face Hub "
                  "(requires internet; replace with LOCAL_BACKBONE_CHECKPOINT before submitting)")
        else:
            print("backbone initialised with random weights (PRETRAINED=False) — "
                  "for pipeline testing only, not for a real submission")
    data_cfg = timm.data.resolve_data_config({}, model=model)
    mean = torch.tensor(data_cfg["mean"]).view(1, 3, 1, 1)
    std = torch.tensor(data_cfg["std"]).view(1, 3, 1, 1)
    if cfg["FREEZE_BACKBONE"]:
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
    return model.to(DEVICE), mean.to(DEVICE), std.to(DEVICE)

backbone, backbone_mean, backbone_std = load_backbone(CONFIG)
embed_dim = backbone.num_features
print(f"backbone embedding dimension : {embed_dim}, frozen: {CONFIG['FREEZE_BACKBONE']}")

# Pristine snapshot of the pretrained weights. Every fine-tuned member restores this before
# training, so members never inherit another member's drifted weights.
original_backbone_state = {k: v.detach().clone() for k, v in backbone.state_dict().items()}

loaded backbone weights from local checkpoint: /kaggle/input/dinov2-weights/dinov2_backbone.pth
backbone embedding dimension : 384, frozen: True


### Offline scoring

Scored runs have no internet, so `pretrained=True` (Hub download) fails there. Run once with internet: the cell below saves the backbone weights to `/kaggle/working/`. Upload that file as a Kaggle dataset, attach it, and set `CONFIG["LOCAL_BACKBONE_CHECKPOINT"]` to its path. The TF-IDF label model is plain scikit-learn trained on the training reports each run, so it needs no external weights.

In [6]:
BACKBONE_CHECKPOINT_OUT = "/kaggle/working/dinov2_backbone.pth"

if CONFIG["PRETRAINED"] and not CONFIG["LOCAL_BACKBONE_CHECKPOINT"]:
    torch.save(backbone.state_dict(), BACKBONE_CHECKPOINT_OUT)
    print(f"backbone weights saved to {BACKBONE_CHECKPOINT_OUT}")
    print("download this file, attach it to this notebook as a Kaggle dataset, then set "
          "CONFIG[\"LOCAL_BACKBONE_CHECKPOINT\"] to its path before a scored submission run.")
else:
    print("skipped saving: either already loading from a local checkpoint "
          "(CONFIG['LOCAL_BACKBONE_CHECKPOINT'] is set), or CONFIG['PRETRAINED'] is False.")

skipped saving: either already loading from a local checkpoint (CONFIG['LOCAL_BACKBONE_CHECKPOINT'] is set), or CONFIG['PRETRAINED'] is False.


Features are extracted one plane at a time so each batch contains same-plane slices. Once the slice and feature caches exist, every later cell runs in seconds.

In [7]:
@torch.no_grad()
def extract_features(slice_tensor, backbone, mean, std, cfg, cache_path):
    # slice_tensor: (N, 3, K, size, size) float32 in [0, 1]. Returns (N, 3 * embed_dim).
    if os.path.exists(cache_path):
        print(f"loading cached features from {cache_path}")
        return np.load(cache_path)

    n_studies, n_planes, k, size, _ = slice_tensor.shape
    embed_dim = backbone.num_features
    batch_size = cfg["FEATURE_BATCH_SIZE"]
    use_amp = DEVICE.type == "cuda"

    # Flatten (plane, study, slice) into one long axis so the backbone always sees batches of
    # a fixed, configurable size, regardless of how many planes or slices-per-plane are used.
    flat_images = slice_tensor.transpose(1, 0, 2, 3, 4).reshape(n_planes * n_studies * k, size, size)
    flat_embeddings = np.zeros((flat_images.shape[0], embed_dim), dtype=np.float32)

    for start in range(0, flat_images.shape[0], batch_size):
        end = min(start + batch_size, flat_images.shape[0])
        batch = torch.from_numpy(flat_images[start:end]).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)
        batch = (batch - mean) / std
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            out = backbone(batch)
        flat_embeddings[start:end] = out.float().cpu().numpy()

    flat_embeddings = flat_embeddings.reshape(n_planes, n_studies, k, embed_dim)
    plane_embeddings = flat_embeddings.mean(axis=2)          # average over the K slices
    study_features = plane_embeddings.transpose(1, 0, 2).reshape(n_studies, n_planes * embed_dim)

    np.save(cache_path, study_features)
    print(f"extracted features for {n_studies} studies, saved to {cache_path}")
    return study_features

train_feature_cache = os.path.join(
    CONFIG["CACHE_DIR"],
    f"train_features_{CONFIG['BACKBONE_NAME']}_sz{CONFIG['IMG_SIZE']}"
    f"_k{CONFIG['SLICES_PER_PLANE']}_lat{int(CONFIG['LATERALITY_CORRECTION'])}.npy")
test_feature_cache = os.path.join(
    CONFIG["CACHE_DIR"],
    f"test_features_{CONFIG['BACKBONE_NAME']}_sz{CONFIG['IMG_SIZE']}"
    f"_k{CONFIG['SLICES_PER_PLANE']}_lat{int(CONFIG['LATERALITY_CORRECTION'])}.npy")

X_train_feat = extract_features(train_slices, backbone, backbone_mean, backbone_std, CONFIG, train_feature_cache)
X_test_feat = extract_features(test_slices, backbone, backbone_mean, backbone_std, CONFIG, test_feature_cache)
print("X_train_feat shape:", X_train_feat.shape, " X_test_feat shape:", X_test_feat.shape)

extracted features for 4407 studies, saved to /kaggle/working/cache/train_features_vit_small_patch14_dinov2_sz224_k3_lat1.npy
extracted features for 3 studies, saved to /kaggle/working/cache/test_features_vit_small_patch14_dinov2_sz224_k3_lat1.npy
X_train_feat shape: (4407, 1152)  X_test_feat shape: (3, 1152)


## 5. Weak labels from reports

Most studies have no direct labels, so a first-pass label is extracted from the report with per-finding keyword patterns and a nearby-negation check: 0.85 for a likely positive, 0.15 for a likely negative, 0.5 (the least informative value under binary cross-entropy) for no match. Gold labels override these and receive weight 10.

The patterns cover only a few languages: on this data they match 29.3% of (study, finding) pairs, so most pairs stay at the 0.5 placeholder.

In [8]:
FINDING_PATTERNS = {
    "ACL": r"\bacl\b|anterior cruciate|ligament.{0,15}crois|ligamento cruzado anterior|kreuzband",
    "MCL": r"\bmcl\b|medial collateral|colateral medial|innenband|ligamento colateral medial",
    "Medial Meniscus": r"medial meniscus|menisco medial|menisque interne|innenmeniskus",
    "Lateral Meniscus": r"lateral meniscus|menisco lateral|menisque externe|aussenmeniskus",
    "Medial OA": r"medial.{0,20}(arthrosis|osteoarthritis|compartment)|artrosis medial",
    "Lateral OA": r"lateral.{0,20}(arthrosis|osteoarthritis|compartment)|artrosis lateral",
    "PF OA": r"patellofemoral.{0,20}(arthrosis|osteoarthritis)|femoropatellar",
    "Effusion": r"effusion|joint fluid|derrame articular|epanchement|gelenkerguss",
    "Synovitis": r"synovitis|synovial|sinovitis|hoffa",
    "Baker's": r"baker|popliteal cyst|quiste popliteo|kyste poplite",
    "Contusion": r"contusion|bone bruise|bone marrow edema|edema.{0,10}medular|knochenmarks?odem",
    "Fracture": r"fracture|fractura|fraktur",
}
NEGATION_WORDS = r"(?:no|not|without|negative for|sin|sans|aucun|nicht|keine?|kein)\W{0,45}$"

def weak_labels_from_report(text):
    normalized = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii").lower()
    states = []
    for finding in TARGETS:
        pattern = FINDING_PATTERNS.get(finding)
        state = 0
        if pattern:
            for match in re.finditer(pattern, normalized):
                preceding_text = normalized[max(0, match.start() - 45):match.start()]
                state = -1 if re.search(NEGATION_WORDS, preceding_text) else 1
                if state == 1:
                    break
        states.append(state)
    return states

weak_states = np.array([weak_labels_from_report(t) for t in train["Report"].fillna("")], dtype=np.int8)
soft_targets = np.where(weak_states == 1, 0.85, np.where(weak_states == -1, 0.15, 0.5)).astype(np.float32)
sample_weights = np.where(weak_states == 0, 0.2, 1.0).astype(np.float32)

gold_mask = has_full_labels.to_numpy()
soft_targets[gold_mask] = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)
sample_weights[gold_mask] = 10.0

print(f"share of (study, finding) pairs the keyword search matched : {(weak_states != 0).mean() * 100:.1f}%")
print(f"directly labelled studies overriding the weak targets      : {int(gold_mask.sum())}")

share of (study, finding) pairs the keyword search matched : 29.3%
directly labelled studies overriding the weak targets      : 58


### 5.1 TF-IDF report classifier

A per-finding logistic regression on character n-gram TF-IDF features, trained on the 58 gold (report, label) pairs. Character n-grams share substrings across related languages ("fracture"/"fraktur", "arthrose"/"arthrosis") without per-language vocabularies or downloaded models. It is evaluated by pooled leave-one-out: each gold study is scored by a model that never saw it.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

def normalize_report_text(text):
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii").lower()

report_texts_normalized = train["Report"].fillna("").map(normalize_report_text)

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=20000, min_df=2)
X_text_all = vectorizer.fit_transform(report_texts_normalized)
X_text_gold = X_text_all[gold_mask]
Y_text_gold = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)

print(f"TF-IDF vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"gold studies available for the text classifier probe : {X_text_gold.shape[0]}")

TF-IDF vocabulary size : 20,000
gold studies available for the text classifier probe : 58


In [10]:
# Pooled leave-one-out evaluation over the gold studies: every study is scored by a classifier
# that never saw it, and all out-of-fold predictions are pooled into one macro-AUC.
n_gold_text = X_text_gold.shape[0]
n_probe_folds_text = n_gold_text if n_gold_text >= 2 else 0  # pooled leave-one-out

if n_probe_folds_text >= 2:
    kfold_text = KFold(n_splits=n_probe_folds_text, shuffle=True, random_state=0)
    oof_text_pred = np.full_like(Y_text_gold, np.nan)
    for j, finding in enumerate(TARGETS):
        column = Y_text_gold[:, j]
        if len(np.unique(column)) < 2:
            continue
        for train_fold_idx, valid_fold_idx in kfold_text.split(X_text_gold):
            if len(np.unique(column[train_fold_idx])) < 2:
                continue
            clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")
            clf.fit(X_text_gold[train_fold_idx], column[train_fold_idx])
            oof_text_pred[valid_fold_idx, j] = clf.predict_proba(X_text_gold[valid_fold_idx])[:, 1]

    print(f"{'finding':20} {'n positive':>11} {'text-only pooled OOF AUC':>25}")
    text_probe_scores = []
    for j, finding in enumerate(TARGETS):
        column = Y_text_gold[:, j]
        valid_mask = ~np.isnan(oof_text_pred[:, j])
        if valid_mask.sum() >= 2 and len(np.unique(column[valid_mask])) == 2:
            auc = roc_auc_score(column[valid_mask], oof_text_pred[valid_mask, j])
            text_probe_scores.append(auc)
            print(f"{finding:20} {int(column.sum()):11d} {auc:25.3f}")
        else:
            print(f"{finding:20} {int(column.sum()):11d} {'n/a':>25}")
    if text_probe_scores:
        print(f"\ntext-only pooled gold macro-AUC : {np.mean(text_probe_scores):.3f} "
              f"({len(text_probe_scores)}/{len(TARGETS)} findings) — keyword search "
              f"coverage for comparison: {100*(weak_states != 0).mean():.1f}% of pairs.")
else:
    print(f"only {n_gold_text} directly labelled studies available; skipping this probe "
          f"(needs at least 2 to fit a fold).")

finding               n positive  text-only pooled OOF AUC
ACL                           24                     0.608
MCL                            9                     0.361
Medial Meniscus               26                     0.662
Lateral Meniscus              23                     0.544
Medial OA                     15                     0.628
Lateral OA                    11                     0.569
PF OA                         21                     0.521
Effusion                      35                     0.545
Synovitis                     27                     0.522
Baker's                       12                     0.458
Contusion                     19                     0.613
Fracture                      18                     0.600

text-only pooled gold macro-AUC : 0.553 (12/12 findings) — keyword search coverage for comparison: 29.3% of pairs.


### 5.2 Adopted weak labels

TF-IDF probabilities replace the keyword labels for every non-gold study; gold studies keep their true labels (out-of-fold text predictions are never used for them, so no label leaks into its own feature).

*Evidence:* training the two-phase head under pooled leave-one-out with each label source gave 0.664 macro-AUC for TF-IDF, 0.583 for keyword labels, and 0.588 for a stack of TF-IDF and a multilingual sentence-embedding classifier. The embedding classifier alone scored 0.540 and would also have needed another set of offline weights, so it is not used.

Both label sets are kept as `soft_targets_keyword` and `soft_targets_tfidf`; `soft_targets` points at the TF-IDF version.

In [11]:
soft_targets_keyword = soft_targets.copy()
sample_weights_keyword = sample_weights.copy()

text_weak_targets = np.full((len(train), len(TARGETS)), 0.5, dtype=np.float32)

gold_positions_for_targets = np.flatnonzero(gold_mask)
for j in range(len(TARGETS)):
    text_weak_targets[gold_positions_for_targets, j] = np.nan_to_num(oof_text_pred[:, j], nan=0.5)

non_gold_positions = np.flatnonzero(~gold_mask)
for j, finding in enumerate(TARGETS):
    column = Y_text_gold[:, j]
    if len(np.unique(column)) < 2:
        continue  # leave this finding at the uninformative 0.5 for non-gold studies
    clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")
    clf.fit(X_text_gold, column)
    text_weak_targets[non_gold_positions, j] = clf.predict_proba(X_text_all[non_gold_positions])[:, 1]

# Adopt as the production targets: TF-IDF-derived probability everywhere, overridden by the
# true label for the 58 directly labelled studies, exactly as the keyword version was.
soft_targets_tfidf = text_weak_targets.copy()
soft_targets_tfidf[gold_mask] = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)

# Every non-gold study now has a data-derived probability, so one uniform weight is used
# instead of the keyword version's match/no-match discount.
sample_weights_tfidf = np.ones(len(train), dtype=np.float32)
sample_weights_tfidf[gold_mask] = 10.0

soft_targets = soft_targets_tfidf
sample_weights = sample_weights_tfidf

print("adopted TF-IDF-derived weak labels for training (see docs/experiments.md for the comparison):")
print(f"  coverage        : 100% of studies now carry a data-derived probability "
      f"(the keyword search covered {100 * (weak_states != 0).mean():.1f}%)")
print(f"  directly labelled studies still overriding with their true label : {int(gold_mask.sum())}")

adopted TF-IDF-derived weak labels for training (see docs/experiments.md for the comparison):
  coverage        : 100% of studies now carry a data-derived probability (the keyword search covered 29.3%)
  directly labelled studies still overriding with their true label : 58


## 6. Leakage-free, stratified folds

Studies with identical report text share weak targets, so folds are assigned by a hash of the report text: duplicates always stay together, and blank-report studies are hashed individually.

A hash is random with respect to the gold labels, and with rare findings that matters: before stratification several findings had no gold positives in some folds (for example Lateral OA and Baker's in fold 0). The gold studies, plus any study sharing identical report text with one, are therefore re-assigned with multi-label iterative stratification (Sechidis et al., 2011), which balances all twelve findings across folds simultaneously by repeatedly placing the rarest remaining finding where it is most needed. It is implemented in numpy to avoid a dependency that may be missing offline. All other studies keep their hash-based fold.

The table shows gold-positive counts per fold before and after; after stratification every finding has at least one positive in every fold.

In [12]:
def report_key(text, study_id):
    # The grouping key used for leakage prevention: identical report text gets the same key
    # (so duplicate reports always land in the same fold), and an empty report is keyed by its
    # own study so distinct blank-report studies are not all forced into one group.
    text = str(text).strip()
    return text if text else f"__empty__{study_id}"

def report_group_id(text, study_id, n_folds):
    key = report_key(text, study_id)
    return int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) % n_folds

fold_id = np.array([report_group_id(t, s, CONFIG["N_FOLDS"])
                     for t, s in zip(train["Report"].fillna(""), train["StudyInstanceUID"])])


def iterative_stratify(label_matrix, n_folds, seed=0):
    # Multi-label iterative stratification (Sechidis, Tsoumakas & Vlahavas, 2011), implemented
    # directly in numpy rather than pulling in scikit-multilearn, so the offline submission
    # notebook never needs a dependency beyond what is already pinned here. Balances every
    # label's positive rate across folds simultaneously, rather than only one label at a time
    # the way sklearn's own StratifiedKFold does -- plain StratifiedKFold has no multi-label
    # mode at all.
    #
    # The idea: repeatedly take the label with the fewest remaining unassigned positive
    # examples (the rarest one left, since it is the easiest to accidentally concentrate into
    # one fold), and assign each of its examples to whichever fold still has the largest
    # remaining "quota" for that label, breaking ties toward whichever fold has the most
    # remaining quota overall. Once every label is exhausted, any leftover examples (all-zero
    # rows) are filled in by whichever fold has the most remaining capacity, to keep fold sizes
    # even.
    rng = np.random.default_rng(seed)
    n_items, n_labels = label_matrix.shape
    unassigned = set(range(n_items))
    fold_of = np.full(n_items, -1, dtype=int)

    desired = np.zeros((n_folds, n_labels), dtype=float)
    for j in range(n_labels):
        desired[:, j] = label_matrix[:, j].sum() / n_folds
    desired_total = np.full(n_folds, n_items / n_folds, dtype=float)

    while unassigned:
        remaining_idx = np.array(sorted(unassigned))
        pos_counts = label_matrix[remaining_idx].sum(axis=0)
        candidate_labels = np.flatnonzero(pos_counts > 0)
        if len(candidate_labels) == 0:
            for i in remaining_idx:
                fold = int(np.argmax(desired_total))
                fold_of[i] = fold
                desired_total[fold] -= 1
                unassigned.discard(i)
            break

        j = candidate_labels[np.argmin(pos_counts[candidate_labels])]
        items_with_j = [i for i in remaining_idx if label_matrix[i, j] == 1]
        rng.shuffle(items_with_j)

        for i in items_with_j:
            max_val = desired[:, j].max()
            tied = np.flatnonzero(desired[:, j] == max_val)
            if len(tied) > 1:
                max_total = desired_total[tied].max()
                tied = tied[desired_total[tied] == max_total]
            fold = int(rng.choice(tied)) if len(tied) > 1 else int(tied[0])

            fold_of[i] = fold
            desired[fold] -= label_matrix[i]
            desired_total[fold] -= 1
            unassigned.discard(i)

    return fold_of


# --- override the gold studies' fold assignment with a stratified one ---
# The hash-based assignment above was never designed to balance anything about the gold
# labels; it is essentially random with respect to them. With findings as imbalanced as these
# (some gold-positive counts are in the single digits), an unlucky random split can leave one
# fold's gold-only phase-2 training pool with zero examples of a rare finding, or concentrate
# most of a rare finding's positive examples into whichever fold happens to hold them out --
# exactly the kind of fold-composition noise this notebook's pooled-CV methodology has been
# trying to eliminate elsewhere.
#
# This only touches the gold studies (and anything sharing identical report text with one of
# them, to preserve the no-leakage property above): every other study keeps its original
# hash-based fold, since there is no gold label to stratify by for the rest of the pool.
gold_positions_6 = np.flatnonzero(gold_mask)
gold_keys_6 = {report_key(t, s) for t, s in zip(train["Report"].fillna("").iloc[gold_positions_6],
                                                 train["StudyInstanceUID"].iloc[gold_positions_6])}
all_keys_6 = np.array([report_key(t, s) for t, s in zip(train["Report"].fillna(""), train["StudyInstanceUID"])])

group_members = {key: np.flatnonzero(all_keys_6 == key) for key in gold_keys_6}
group_keys_ordered = sorted(group_members)  # fixed order so fold_of below lines up with it

gold_targets_6 = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)
gold_position_of_study = {pos: row for row, pos in enumerate(gold_positions_6)}

group_label_vectors = []
for key in group_keys_ordered:
    members = group_members[key]
    gold_members = [m for m in members if m in gold_position_of_study]
    rows = [gold_targets_6[gold_position_of_study[m]] for m in gold_members]
    group_label_vectors.append(np.mean(rows, axis=0) if rows else np.zeros(len(TARGETS)))
group_label_matrix = np.round(np.stack(group_label_vectors)).astype(int)

group_fold_of = iterative_stratify(group_label_matrix, CONFIG["N_FOLDS"], seed=0)

before_fold_id_gold = fold_id.copy()
for key, assigned_fold in zip(group_keys_ordered, group_fold_of):
    fold_id[group_members[key]] = assigned_fold

train_idx = np.flatnonzero(fold_id != CONFIG["VALID_FOLD"])
valid_idx = np.flatnonzero(fold_id == CONFIG["VALID_FOLD"])

print(f"training examples   : {len(train_idx):,}")
print(f"validation examples : {len(valid_idx):,}")
print(f"directly labelled studies in the validation fold : {int(gold_mask[valid_idx].sum())}")

print(f"\n{'finding':20} {'total':>6} " + " ".join(f"{'before f' + str(f):>9}" for f in range(CONFIG['N_FOLDS']))
      + " || " + " ".join(f"{'after f' + str(f):>8}" for f in range(CONFIG['N_FOLDS'])))
for j, finding in enumerate(TARGETS):
    before_counts = [int(gold_targets_6[(before_fold_id_gold[gold_positions_6] == f), j].sum())
                      for f in range(CONFIG["N_FOLDS"])]
    after_counts = [int(gold_targets_6[(fold_id[gold_positions_6] == f), j].sum())
                     for f in range(CONFIG["N_FOLDS"])]
    total = int(gold_targets_6[:, j].sum())
    print(f"{finding:20} {total:6d} " + " ".join(f"{c:9d}" for c in before_counts)
          + " || " + " ".join(f"{c:8d}" for c in after_counts))

training examples   : 3,537
validation examples : 870
directly labelled studies in the validation fold : 12

finding               total before f0 before f1 before f2 before f3 before f4 || after f0 after f1 after f2 after f3 after f4
ACL                      24         2         7         5         1         9 ||        5        4        5        5        5
MCL                       9         2         2         1         0         4 ||        1        2        2        2        2
Medial Meniscus          26         1         6         4         3        12 ||        4        5        5        5        7
Lateral Meniscus         23         1         5         2         3        12 ||        5        5        5        4        4
Medial OA                15         2         3         1         2         7 ||        3        3        3        3        3
Lateral OA               11         0         2         0         1         8 ||        3        2        2        2        2
PF OA    

## 7. Two-phase head

The head is an MLP (LayerNorm, hidden layer, dropout) on the cached features.

Training on weak and gold targets jointly for the whole schedule fails: weak-label validation AUC kept rising while gold validation AUC fell, because the weak-label loss dominates and the model learns to agree with the labeller rather than the finding. Training is therefore split in two:

1. **Phase 1:** a short warm-up on the full weighted weak-plus-gold pool.
2. **Phase 2:** the same weights continue on gold studies only, at a lower learning rate.

*Evidence (pooled leave-one-out on the gold studies):* the two-phase head reaches 0.658 macro-AUC against 0.609 for a plain logistic-regression probe on the same frozen features.

With one held-out study an AUC is undefined, so leave-one-out runs train for the fixed schedule and score once. The production folds select the best epoch on the validation fold when a score is available and otherwise keep the end-of-schedule weights.

In [13]:
class KneeHead(nn.Module):
    def __init__(self, in_dim, n_targets, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_targets),
        )

    def forward(self, x):
        return self.net(x)

def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32)

def macro_auc(y_true, y_pred):
    scores = []
    for j in range(y_true.shape[1]):
        column = y_true[:, j]
        if len(np.unique(column)) < 2:
            continue
        scores.append(roc_auc_score(column, y_pred[:, j]))
    return float(np.mean(scores)) if scores else float("nan")

def macro_auc_from_weak_states(states, y_pred):
    # AUC against the weak labels, restricted per finding to studies the keyword search was
    # confident about (state == 1 or -1). Unmatched studies (state == 0) carry no reliable
    # sign and are excluded rather than being scored against the uninformative 0.5 placeholder
    # value, which is not a valid binary target for roc_auc_score in the first place.
    scores = []
    for j in range(states.shape[1]):
        column = states[:, j]
        confident = column != 0
        if confident.sum() < 2:
            continue
        y_true = (column[confident] == 1).astype(int)
        if len(np.unique(y_true)) < 2:
            continue
        scores.append(roc_auc_score(y_true, y_pred[confident, j]))
    return float(np.mean(scores)) if scores else float("nan")

def train_two_phase_head(cfg, feat_dim, n_targets,
                          phase1_X, phase1_Y_soft, phase1_W,
                          phase2_X, phase2_Y_hard,
                          eval_X, eval_weak_states, eval_gold_mask, eval_Y_gold,
                          min_gold_for_selection=5, verbose=True):
    # Runs the warm-up-then-gold-fine-tune schedule described above and returns the best
    # head found, together with the score it was selected on and whether a real improvement
    # was ever observed (as opposed to falling back to the initial weights).
    head = KneeHead(feat_dim, n_targets, cfg["HEAD_HIDDEN"], cfg["HEAD_DROPOUT"]).to(DEVICE)
    best_score = -1.0
    best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
    used_fallback_state = True

    def evaluate():
        head.eval()
        with torch.no_grad():
            pred = torch.sigmoid(head(eval_X)).cpu().numpy()
        weak_auc = macro_auc_from_weak_states(eval_weak_states, pred)
        gold_auc = (macro_auc(eval_Y_gold, pred[eval_gold_mask])
                    if eval_gold_mask.sum() >= min_gold_for_selection else float("nan"))
        return weak_auc, gold_auc

    def maybe_keep(selection_score):
        nonlocal best_score, best_state, used_fallback_state
        if not np.isnan(selection_score) and selection_score > best_score:
            best_score = selection_score
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            used_fallback_state = False

    # --- phase 1: brief warm-up on the full weak-and-gold weighted pool ---
    n1 = phase1_X.shape[0]
    batch1 = min(cfg["HEAD_BATCH_SIZE"], n1)
    optimizer = torch.optim.AdamW(head.parameters(), lr=cfg["PHASE1_LR"], weight_decay=cfg["HEAD_WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(cfg["PHASE1_EPOCHS"], 1))
    for epoch in range(cfg["PHASE1_EPOCHS"]):
        head.train()
        permutation = torch.randperm(n1, device=DEVICE)
        for start in range(0, n1, batch1):
            batch_idx = permutation[start:start + batch1]
            logits = head(phase1_X[batch_idx])
            loss_per_element = F.binary_cross_entropy_with_logits(logits, phase1_Y_soft[batch_idx], reduction="none")
            loss = (loss_per_element * phase1_W[batch_idx].unsqueeze(1)).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        scheduler.step()
        weak_auc, gold_auc = evaluate()
        maybe_keep(gold_auc if not np.isnan(gold_auc) else weak_auc)
        if verbose and (epoch % 10 == 0 or epoch == cfg["PHASE1_EPOCHS"] - 1):
            gold_str = f"{gold_auc:.4f}" if not np.isnan(gold_auc) else "n/a"
            weak_str = f"{weak_auc:.4f}" if not np.isnan(weak_auc) else "n/a"
            print(f"[phase 1] epoch {epoch:3d}  weak-val AUC {weak_str}  gold-val AUC {gold_str}")

    # --- phase 2: fine-tune using only the directly labelled studies, at a low learning rate ---
    n2 = phase2_X.shape[0]
    if n2 >= 2:
        batch2 = min(cfg["HEAD_BATCH_SIZE"], n2)
        optimizer = torch.optim.AdamW(head.parameters(), lr=cfg["PHASE2_LR"], weight_decay=cfg["HEAD_WEIGHT_DECAY"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(cfg["PHASE2_EPOCHS"], 1))
        for epoch in range(cfg["PHASE2_EPOCHS"]):
            head.train()
            permutation = torch.randperm(n2, device=DEVICE)
            for start in range(0, n2, batch2):
                batch_idx = permutation[start:start + batch2]
                logits = head(phase2_X[batch_idx])
                loss = F.binary_cross_entropy_with_logits(logits, phase2_Y_hard[batch_idx])
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()
            weak_auc, gold_auc = evaluate()
            maybe_keep(gold_auc if not np.isnan(gold_auc) else weak_auc)
            if verbose and (epoch % 20 == 0 or epoch == cfg["PHASE2_EPOCHS"] - 1):
                gold_str = f"{gold_auc:.4f}" if not np.isnan(gold_auc) else "n/a"
                weak_str = f"{weak_auc:.4f}" if not np.isnan(weak_auc) else "n/a"
                print(f"[phase 2] epoch {epoch:3d}  weak-val AUC {weak_str}  gold-val AUC {gold_str}")
    elif verbose:
        print("phase 2 skipped: fewer than two directly labelled studies available to fine-tune on")

    if used_fallback_state:
        # No epoch's held-out score was ever usable for selection -- for example under
        # leave-one-out cross-validation, where a single held-out study can never supply both
        # classes for any finding, so gold_auc and weak_auc come back NaN every single epoch,
        # every single fold. best_state was initialised to this function's literal random,
        # untrained starting weights; falling back to that would silently discard everything
        # phase 1 and phase 2 just trained. The head already holds the fully-trained
        # end-of-schedule weights at this point, which are strictly more informative than
        # random initialisation even though there is no way to confirm on this fold that they
        # improved anything.
        best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}

    head.load_state_dict(best_state)
    return head, best_score, used_fallback_state

gold_targets_full = np.full((len(train), len(TARGETS)), np.nan, dtype=np.float32)
gold_targets_full[gold_mask] = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)


## 8. Adapting the backbone

A frozen natural-image backbone is a plausible bottleneck, so two ways of adapting it are used as additional ensemble members, trained only on gold studies (phase 2 labels are the only ones known to be correct):

- **Last-block fine-tuning:** the last `N_TRAINABLE_BACKBONE_BLOCKS` transformer blocks and the final norm are unfrozen at a learning rate 10x below the head's. One block is used because an earlier comparison found two blocks overfit more.
- **LoRA:** the backbone stays frozen and low-rank adapters are added to the attention projections of the last `N_LORA_BLOCKS` blocks, training about 4% as many parameters as one fully unfrozen block.

These use the raw cached slices rather than cached features, so each member re-extracts test features through its own backbone.

*Evidence (pooled leave-one-out, single seed):* frozen features 0.664, last-block fine-tuning 0.644, LoRA 0.619 macro-AUC. Adaptation did not beat the frozen head in validation, so these members are weighted below the frozen group rather than trusted blindly; they are included for diversity, and `docs/experiments.md` lists what remains untested.

In [14]:
def set_backbone_trainable_blocks(backbone, n_blocks):
    # Freezes every parameter, then unfreezes the last n_blocks transformer blocks plus the
    # final normalisation layer(s), if the backbone exposes the attributes timm's ViT
    # implementations typically use. Leaves the backbone fully frozen when n_blocks <= 0.
    for p in backbone.parameters():
        p.requires_grad_(False)
    if n_blocks <= 0:
        return
    blocks = getattr(backbone, "blocks", None)
    if blocks is None:
        raise AttributeError("backbone has no .blocks attribute; cannot unfreeze by block count")
    for block in list(blocks)[-n_blocks:]:
        for p in block.parameters():
            p.requires_grad_(True)
    for attr in ("norm", "fc_norm"):
        module = getattr(backbone, attr, None)
        if module is not None:
            for p in module.parameters():
                p.requires_grad_(True)

def encode_studies_with_grad(raw_slices, backbone, mean, std):
    # Differentiable counterpart to the cached feature extraction: takes a raw
    # (n_studies, 3, K, size, size) array or tensor and returns (n_studies, 3 * embed_dim)
    # embeddings, keeping gradients so the unfrozen backbone blocks can be updated. Used only
    # on the small gold-only batches below, never on the full imaging dataset.
    x = torch.from_numpy(raw_slices) if isinstance(raw_slices, np.ndarray) else raw_slices
    x = x.to(DEVICE)
    n_studies, n_planes, k, size, _ = x.shape
    flat = x.reshape(n_studies * n_planes * k, size, size)
    imgs = flat.unsqueeze(1).repeat(1, 3, 1, 1)
    imgs = (imgs - mean) / std
    use_amp = DEVICE.type == "cuda"
    with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
        out = backbone(imgs)
    embed_dim = out.shape[-1]
    out = out.reshape(n_studies, n_planes, k, embed_dim).mean(dim=2)
    return out.reshape(n_studies, n_planes * embed_dim)

set_backbone_trainable_blocks(backbone, 0)  # shared backbone stays frozen until a member trains


### Augmentation and LoRA building blocks

`augment_raw_batch` applies small rotations and translations, intensity scale/bias jitter and mild Gaussian noise, one draw per study shared across its planes and slices. Random flips are off, since they would undo the laterality mirror. Augmentation touches training batches only.

*Evidence:* in leave-one-out CV augmentation left macro-AUC unchanged (0.652 vs 0.652) while moving individual findings both ways (Synovitis +0.062, Baker's -0.096), so it is neutral on average rather than a demonstrated gain.

`LoRALinear` wraps a frozen linear layer with a zero-initialised low-rank update, so a wrapped layer initially reproduces the original output. `build_lora_backbone` builds a fresh backbone per member because wrapping changes state-dict keys.

In [15]:
from scipy import ndimage

def augment_raw_batch(raw_batch, cfg, rng):
    # raw_batch: (B, 3, K, size, size) float32 in [0, 1]. Draws one random transform per study,
    # applied identically across that study's own planes and slices, since a shared physical
    # cause (patient positioning, scanner gain) would affect the whole scan the same way, not a
    # fresh independent draw per slice. Returns a new array; raw_batch itself is left untouched.
    augmented = raw_batch.astype(np.float32, copy=True)
    size = raw_batch.shape[-1]
    shift_limit = cfg["AUGMENT_TRANSLATE_FRAC"] * size

    for b in range(augmented.shape[0]):
        angle = rng.uniform(-cfg["AUGMENT_ROTATION_DEG"], cfg["AUGMENT_ROTATION_DEG"])
        shift_y = rng.uniform(-shift_limit, shift_limit)
        shift_x = rng.uniform(-shift_limit, shift_limit)
        intensity_scale = 1.0 + rng.uniform(-cfg["AUGMENT_INTENSITY_SCALE"], cfg["AUGMENT_INTENSITY_SCALE"])
        intensity_bias = rng.uniform(-cfg["AUGMENT_INTENSITY_SCALE"], cfg["AUGMENT_INTENSITY_SCALE"]) * 0.5
        do_flip = cfg["AUGMENT_FLIP_PROB"] > 0 and rng.random() < cfg["AUGMENT_FLIP_PROB"]

        for plane_idx in range(augmented.shape[1]):
            for slice_idx in range(augmented.shape[2]):
                img = augmented[b, plane_idx, slice_idx]
                if angle != 0.0:
                    img = ndimage.rotate(img, angle, reshape=False, order=1, mode="nearest")
                if shift_y != 0.0 or shift_x != 0.0:
                    img = ndimage.shift(img, (shift_y, shift_x), order=1, mode="nearest")
                img = img * intensity_scale + intensity_bias
                if do_flip:
                    img = img[:, ::-1]
                augmented[b, plane_idx, slice_idx] = img

        noise = rng.normal(0.0, cfg["AUGMENT_NOISE_STD"], augmented[b].shape).astype(np.float32)
        augmented[b] = augmented[b] + noise

    return np.clip(augmented, 0.0, 1.0).astype(np.float32)


In [16]:
class LoRALinear(nn.Module):
    # Wraps a frozen nn.Linear, adding a small trainable low-rank correction alongside it:
    # output = base(x) + scaling * (x @ lora_A.T @ lora_B.T), with lora_A: (rank, in_features)
    # and lora_B: (out_features, rank). lora_B is zero-initialised, so a freshly-wrapped layer
    # computes identically to the original Linear until any training happens -- wrapping a
    # layer never changes its output on its own.
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.lora_A = nn.Parameter(torch.randn(rank, base_linear.in_features) * rank ** -0.5)
        self.lora_B = nn.Parameter(torch.zeros(base_linear.out_features, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scaling * ((x @ self.lora_A.T) @ self.lora_B.T)


def build_lora_backbone(cfg):
    # Builds a fresh, pristine backbone (same architecture and pretrained weights as the
    # shared `backbone` object, loaded from the same original_backbone_state snapshot) and
    # wraps the attention qkv and output projections of its last cfg["N_LORA_BLOCKS"]
    # transformer blocks with LoRALinear adapters. A fresh instance is used because wrapping
    # changes the module structure (state-dict keys become "base.weight"), which the
    # load_state_dict reset used for the shared backbone cannot undo.
    #
    # LoRA is applied only to attn.qkv and attn.proj (the attention projections), not the MLP
    # layers, which is the standard scope from the original LoRA paper and keeps the parameter
    # count low; CONFIG["N_LORA_BLOCKS"] can cover more blocks than full fine-tuning typically
    # would, since each block's LoRA adapters are a small fraction of that block's own
    # parameter count.
    fresh_backbone = timm.create_model(cfg["BACKBONE_NAME"], pretrained=False, num_classes=0,
                                        img_size=cfg["IMG_SIZE"], dynamic_img_size=True)
    fresh_backbone.load_state_dict(original_backbone_state)
    fresh_backbone.to(DEVICE)
    for p in fresh_backbone.parameters():
        p.requires_grad_(False)

    lora_params = []
    blocks = list(fresh_backbone.blocks)[-cfg["N_LORA_BLOCKS"]:] if cfg["N_LORA_BLOCKS"] > 0 else []
    for block in blocks:
        block.attn.qkv = LoRALinear(block.attn.qkv, cfg["LORA_RANK"], cfg["LORA_ALPHA"]).to(DEVICE)
        block.attn.proj = LoRALinear(block.attn.proj, cfg["LORA_RANK"], cfg["LORA_ALPHA"]).to(DEVICE)
        lora_params += [block.attn.qkv.lora_A, block.attn.qkv.lora_B,
                         block.attn.proj.lora_A, block.attn.proj.lora_B]
    return fresh_backbone, lora_params


## 9. Ensemble and test prediction

Per fold, three groups of members are built and averaged within their group:

- **frozen:** two-phase heads on cached features, with extra seeds (cheap, runs first);
- **finetuned:** head plus last-block fine-tuned backbone, per seed;
- **lora:** head plus LoRA adapters on a fresh backbone, per seed.

Groups are combined with `ENSEMBLE_GROUP_WEIGHTS` (0.5 / 0.25 / 0.25). The weights are a hedge toward the group that validated best, not a tuned optimum.

Optional members train in rounds across folds, and are skipped once `ENSEMBLE_TIME_BUDGET_SEC` has elapsed, so a slow run degrades gracefully instead of timing out. Test studies go through each member in chunks, so memory does not grow with test-set size.

In [17]:
def predict_test_through(backbone_model, head_model, chunk=None):
    # Chunked, gradient-free prediction of the whole test set through a (possibly adapted) backbone.
    chunk = chunk or CONFIG["TEST_PREDICT_CHUNK"]
    backbone_model.eval()
    head_model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, test_slices.shape[0], chunk):
            raw = test_slices[start:start + chunk]
            embeddings = encode_studies_with_grad(raw, backbone_model, backbone_mean, backbone_std)
            outputs.append(torch.sigmoid(head_model(embeddings)).cpu().numpy())
    return np.concatenate(outputs, axis=0)


def train_backbone_member(fold_head_base, fold_phase2_idx, seed, cfg, kind, save_prefix=None):
    # kind "full": unfreeze the last N_TRAINABLE_BACKBONE_BLOCKS of the shared backbone (reset to the
    # pristine snapshot first). kind "lora": fresh backbone with LoRA adapters. Both start the head
    # from the fold's frozen-feature head, train on that fold's gold studies for the fixed configured
    # number of epochs, and return test predictions. Global RNG state is restored afterwards.
    torch_rng_state = torch.get_rng_state()
    numpy_rng_state = np.random.get_state()
    torch.manual_seed(seed)
    np.random.seed(seed)
    aug_rng = np.random.default_rng(cfg["AUGMENT_SEED"] + seed) if cfg["AUGMENT_FINETUNING"] else None
    model = None
    try:
        if kind == "full":
            model = backbone
            model.load_state_dict(original_backbone_state)
            set_backbone_trainable_blocks(model, cfg["N_TRAINABLE_BACKBONE_BLOCKS"])
            backbone_params = [p for p in model.parameters() if p.requires_grad]
            backbone_lr = cfg["BACKBONE_FINETUNE_LR"]
        elif kind == "lora":
            model, backbone_params = build_lora_backbone(cfg)
            backbone_lr = cfg["LORA_LR"]
        else:
            raise ValueError(kind)

        member_head = KneeHead(X_train_feat.shape[1], len(TARGETS),
                                cfg["HEAD_HIDDEN"], cfg["HEAD_DROPOUT"]).to(DEVICE)
        member_head.load_state_dict(fold_head_base.state_dict())

        phase2_raw = train_slices[fold_phase2_idx]
        phase2_targets = to_tensor(gold_targets_full[fold_phase2_idx]).to(DEVICE)
        optimizer = torch.optim.AdamW([
            {"params": member_head.parameters(), "lr": cfg["PHASE2_LR"]},
            {"params": backbone_params, "lr": backbone_lr},
        ], weight_decay=cfg["HEAD_WEIGHT_DECAY"])

        n2 = phase2_raw.shape[0]
        batch_size = min(cfg["BACKBONE_FINETUNE_BATCH_SIZE"], max(n2, 1))
        if n2 >= 2:
            for epoch in range(cfg["BACKBONE_FINETUNE_EPOCHS"]):
                model.train()
                member_head.train()
                permutation = np.random.permutation(n2)
                for start in range(0, n2, batch_size):
                    batch_idx = permutation[start:start + batch_size]
                    batch_raw = phase2_raw[batch_idx]
                    if cfg["AUGMENT_FINETUNING"]:
                        batch_raw = augment_raw_batch(batch_raw, cfg, aug_rng)
                    embeddings = encode_studies_with_grad(batch_raw, model, backbone_mean, backbone_std)
                    loss = F.binary_cross_entropy_with_logits(member_head(embeddings), phase2_targets[batch_idx])
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

        test_pred = predict_test_through(model, member_head)

        if save_prefix is not None and kind == "full":
            trainable_names = {name for name, p in model.named_parameters() if p.requires_grad}
            torch.save({k: v for k, v in model.state_dict().items() if k in trainable_names},
                       f"{save_prefix}_backbone_delta.pth")
            torch.save(member_head.state_dict(), f"{save_prefix}_head.pth")
        return test_pred
    finally:
        if kind == "full":
            backbone.load_state_dict(original_backbone_state)
            set_backbone_trainable_blocks(backbone, 0)
        else:
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        torch.set_rng_state(torch_rng_state)
        np.random.set_state(numpy_rng_state)


def over_budget():
    return (time.time() - NOTEBOOK_START) > CONFIG["ENSEMBLE_TIME_BUDGET_SEC"]


ENSEMBLE_DIR = "/kaggle/working/ensemble"
if CONFIG["SAVE_ENSEMBLE_WEIGHTS"]:
    os.makedirs(ENSEMBLE_DIR, exist_ok=True)

member_preds = {"frozen": [], "finetuned": [], "lora": []}
fold_heads, fold_phase2_indices, fold_selection_scores = [], [], []

# --- stage 1: frozen-feature heads, one per fold (always runs) ---
for fold in range(CONFIG["N_FOLDS"]):
    fold_train_idx = np.flatnonzero(fold_id != fold)
    fold_valid_idx = np.flatnonzero(fold_id == fold)

    fold_X_train = to_tensor(X_train_feat[fold_train_idx]).to(DEVICE)
    fold_Y_train = to_tensor(soft_targets[fold_train_idx]).to(DEVICE)
    fold_W_train = to_tensor(sample_weights[fold_train_idx]).to(DEVICE)

    fold_phase2_idx = fold_train_idx[gold_mask[fold_train_idx]]
    fold_X_phase2 = to_tensor(X_train_feat[fold_phase2_idx]).to(DEVICE)
    fold_Y_phase2 = to_tensor(gold_targets_full[fold_phase2_idx]).to(DEVICE)

    fold_X_valid = to_tensor(X_train_feat[fold_valid_idx]).to(DEVICE)
    fold_weak_states_valid = weak_states[fold_valid_idx]
    fold_valid_gold_mask = gold_mask[fold_valid_idx]
    fold_Y_valid_gold = gold_targets_full[fold_valid_idx][fold_valid_gold_mask]

    fold_head, fold_score, fold_used_fallback = train_two_phase_head(
        CONFIG, X_train_feat.shape[1], len(TARGETS),
        fold_X_train, fold_Y_train, fold_W_train,
        fold_X_phase2, fold_Y_phase2,
        fold_X_valid, fold_weak_states_valid, fold_valid_gold_mask, fold_Y_valid_gold,
        verbose=False,
    )
    fold_head.eval()
    with torch.no_grad():
        member_preds["frozen"].append(
            torch.sigmoid(fold_head(to_tensor(X_test_feat).to(DEVICE))).cpu().numpy())
    fold_heads.append(fold_head)
    fold_phase2_indices.append(fold_phase2_idx)
    fold_selection_scores.append(fold_score)
    if CONFIG["SAVE_ENSEMBLE_WEIGHTS"]:
        torch.save(fold_head.state_dict(), f"{ENSEMBLE_DIR}/frozen_head_fold{fold}.pth")
    print(f"fold {fold + 1}/{CONFIG['N_FOLDS']} frozen head trained, selection score {fold_score:.4f}"
          + (" (no epoch score was measurable; kept end-of-schedule weights)" if fold_used_fallback else ""))

print(f"\naverage selection score across folds (frozen stage) : {np.mean(fold_selection_scores):.4f}")

# --- extra frozen-head seeds: same folds, different head initialisation and batch order ---
# Cheap variance reduction (heads train on cached features in about a minute). These join the
# "frozen" group's average only; the seed-less heads above remain the base heads that the
# fine-tuned and LoRA members start from, so those members are unchanged.
for extra_seed in CONFIG["EXTRA_FROZEN_HEAD_SEEDS"]:
    for fold in range(CONFIG["N_FOLDS"]):
        torch.manual_seed(extra_seed * 1000 + fold)
        np.random.seed(extra_seed * 1000 + fold)
        e_train_idx = np.flatnonzero(fold_id != fold)
        e_valid_idx = np.flatnonzero(fold_id == fold)
        e_phase2_idx = e_train_idx[gold_mask[e_train_idx]]
        e_valid_gold_mask = gold_mask[e_valid_idx]
        e_head, _, _ = train_two_phase_head(
            CONFIG, X_train_feat.shape[1], len(TARGETS),
            to_tensor(X_train_feat[e_train_idx]).to(DEVICE),
            to_tensor(soft_targets[e_train_idx]).to(DEVICE),
            to_tensor(sample_weights[e_train_idx]).to(DEVICE),
            to_tensor(X_train_feat[e_phase2_idx]).to(DEVICE),
            to_tensor(gold_targets_full[e_phase2_idx]).to(DEVICE),
            to_tensor(X_train_feat[e_valid_idx]).to(DEVICE), weak_states[e_valid_idx],
            e_valid_gold_mask, gold_targets_full[e_valid_idx][e_valid_gold_mask],
            verbose=False,
        )
        e_head.eval()
        with torch.no_grad():
            member_preds["frozen"].append(
                torch.sigmoid(e_head(to_tensor(X_test_feat).to(DEVICE))).cpu().numpy())
    print(f"extra frozen-head seed {extra_seed} done for all folds "
          f"(elapsed {(time.time() - NOTEBOOK_START) / 3600:.2f} h)")

# --- stage 2: optional backbone-adapting members, in balanced rounds ---
n_rounds = max(len(CONFIG["FINETUNE_SEEDS"]) if CONFIG["INCLUDE_BACKBONE_FINETUNE_IN_SUBMISSION"] else 0,
               len(CONFIG["LORA_SEEDS"]) if CONFIG["INCLUDE_LORA_MEMBERS"] else 0)
stopped_early = False
for round_i in range(n_rounds):
    for fold in range(CONFIG["N_FOLDS"]):
        if CONFIG["INCLUDE_BACKBONE_FINETUNE_IN_SUBMISSION"] and round_i < len(CONFIG["FINETUNE_SEEDS"]):
            if over_budget():
                stopped_early = True
                break
            seed = CONFIG["FINETUNE_SEEDS"][round_i]
            prefix = f"{ENSEMBLE_DIR}/finetuned_fold{fold}_seed{seed}" if CONFIG["SAVE_ENSEMBLE_WEIGHTS"] else None
            t0 = time.time()
            member_preds["finetuned"].append(
                train_backbone_member(fold_heads[fold], fold_phase2_indices[fold], seed, CONFIG, "full", prefix))
            print(f"  fold {fold + 1}, fine-tuned seed {seed} done in {(time.time() - t0) / 60:.1f} min "
                  f"(elapsed {(time.time() - NOTEBOOK_START) / 3600:.2f} h)")
        if CONFIG["INCLUDE_LORA_MEMBERS"] and round_i < len(CONFIG["LORA_SEEDS"]):
            if over_budget():
                stopped_early = True
                break
            seed = CONFIG["LORA_SEEDS"][round_i]
            t0 = time.time()
            member_preds["lora"].append(
                train_backbone_member(fold_heads[fold], fold_phase2_indices[fold], seed, CONFIG, "lora"))
            print(f"  fold {fold + 1}, LoRA seed {seed} done in {(time.time() - t0) / 60:.1f} min "
                  f"(elapsed {(time.time() - NOTEBOOK_START) / 3600:.2f} h)")
    if stopped_early:
        print(f"\ntime budget of {CONFIG['ENSEMBLE_TIME_BUDGET_SEC'] / 3600:.1f} h reached; "
              f"skipping the remaining optional members.")
        break

# --- combine: average within each group, then weight the groups that actually have members ---
group_means = {g: np.mean(p, axis=0) for g, p in member_preds.items() if len(p) > 0}
total_weight = sum(CONFIG["ENSEMBLE_GROUP_WEIGHTS"][g] for g in group_means)
ensemble_test_pred = sum(CONFIG["ENSEMBLE_GROUP_WEIGHTS"][g] * group_means[g] for g in group_means) / total_weight
assert np.isfinite(ensemble_test_pred).all(), "non-finite ensemble predictions"
print("\nensemble composition: " + ", ".join(
    f"{g}={len(member_preds[g])} members (weight {CONFIG['ENSEMBLE_GROUP_WEIGHTS'][g] / total_weight:.2f})"
    for g in group_means))


fold 1/5 frozen head trained, selection score 0.7636
fold 2/5 frozen head trained, selection score 0.7026
fold 3/5 frozen head trained, selection score 0.7608
fold 4/5 frozen head trained, selection score 0.6974
fold 5/5 frozen head trained, selection score 0.7105

average selection score across folds (frozen stage) : 0.7270
extra frozen-head seed 11 done for all folds (elapsed 0.56 h)
extra frozen-head seed 12 done for all folds (elapsed 0.58 h)
  fold 1, fine-tuned seed 1 done in 3.9 min (elapsed 0.64 h)
  fold 1, LoRA seed 1 done in 4.0 min (elapsed 0.71 h)
  fold 2, fine-tuned seed 1 done in 3.9 min (elapsed 0.77 h)
  fold 2, LoRA seed 1 done in 4.0 min (elapsed 0.84 h)
  fold 3, fine-tuned seed 1 done in 4.0 min (elapsed 0.91 h)
  fold 3, LoRA seed 1 done in 4.1 min (elapsed 0.98 h)
  fold 4, fine-tuned seed 1 done in 3.9 min (elapsed 1.04 h)
  fold 4, LoRA seed 1 done in 4.0 min (elapsed 1.11 h)
  fold 5, fine-tuned seed 1 done in 4.0 min (elapsed 1.18 h)
  fold 5, LoRA seed 1 do

## 10. Submission

Predictions are converted to per-finding percentile ranks. ROC-AUC depends only on ordering within each finding, so this does not change the score but makes output easy to sanity-check across findings with different prevalence.

In [18]:
submission = sample_submission.copy()
rank_pct = pd.DataFrame(ensemble_test_pred, columns=TARGETS).rank(pct=True)
submission[TARGETS] = rank_pct.to_numpy()
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("\nsubmission saved to /kaggle/working/submission.csv")

                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  1.000000  0.333333   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.666667  0.666667   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.333333  1.000000   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         1.000000          0.666667   1.000000    0.666667  1.000000   
1         0.666667          1.000000   0.666667    1.000000  0.333333   
2         0.333333          0.333333   0.333333    0.333333  0.666667   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  1.000000   1.000000  1.000000   1.000000  1.000000  
1  0.666667   0.333333  0.666667   0.666667  0.666667  
2  0.333333   0.666667  0.333333   0.333333  0.333333  

submission saved to /kaggle/working/submission.csv


In [19]:
import shutil

print(f"total wall-clock : {(time.time() - NOTEBOOK_START) / 3600:.2f} h")
if CONFIG["CLEAN_CACHE_AT_END"] and os.path.isdir(CONFIG["CACHE_DIR"]):
    freed = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(CONFIG["CACHE_DIR"]) for f in fs)
    shutil.rmtree(CONFIG["CACHE_DIR"], ignore_errors=True)
    print(f"removed {CONFIG['CACHE_DIR']} ({freed / 1e9:.2f} GB) so the saved version only contains the outputs.")


total wall-clock : 1.89 h
removed /kaggle/working/cache (2.01 GB) so the saved version only contains the outputs.
